# ERA V5 Session 12 — 32 Virtual GPU ZeRO Simulation

Educational simulator — not physical GPU benchmarks.

## 02. Assignment objective

Compare baseline DP, ZeRO-1/2/3 on **32 virtual GPUs** with explicit memory, communication, and compute accounting.

## 03. Conceptual model

Each rank is a `VirtualGPU` object with ownership sets for parameters, gradients, and optimizer state.

## 04. Configuration

In [ ]:
import sys
from pathlib import Path
ROOT = Path.cwd()
for candidate in (ROOT, ROOT.parent):
    if (candidate / 'src').exists():
        ROOT = candidate
        break
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import json
import matplotlib.pyplot as plt

from src.config import SimConfig, WORLD_SIZE, GPUS_PER_NODE, NUM_NODES
from src.virtual_gpu import make_cluster
from src.model import DemoModel
from src.memory import MemoryAccounting
from src.simulation import run_strategy
from src.experiments import run_all_experiments, scaling_sweep, save_records
from src.visualization import generate_all_plots

cfg = SimConfig()
print('WORLD_SIZE', WORLD_SIZE, 'nodes', NUM_NODES, 'gpus/node', GPUS_PER_NODE)
print('Simulation assumption bandwidth (Gbps):', cfg.intra_node_bandwidth_gbps, cfg.inter_node_bandwidth_gbps)

## 05. Create 32 virtual GPUs

In [ ]:
gpus = make_cluster(cfg)
assert len(gpus) == 32
for g in gpus[:4]:
    print(f'rank={g.rank} node={g.node_id}')

## 06. Demo model

In [ ]:
model = DemoModel.build(cfg)
print('logical_parameter_count', model.logical_parameter_count)

## 07. Memory model

In [ ]:
from src.virtual_gpu import assign_replicated
assign_replicated(gpus, cfg.parameter_count)
mem = MemoryAccounting(cfg, gpus[0])
bd = mem.breakdown()
print('parameter_memory', bd.parameters)
print('optimizer_memory (Adam m+v FP32)', bd.optimizer)
print('peak', bd.peak)

## 08. Baseline Data Parallel

In [ ]:
m_base = run_strategy('baseline_dp', cfg)
print(m_base)

**OBSERVATION:**

On the default 1M-parameter run, baseline `peak_memory_per_gpu` is **16,078,884** bytes and every rank’s ownership map is the full parameter range (`assign_replicated`). `communication_bytes` for the gradient all-reduce is **4,000,000**.

**QUESTION:** Why does every GPU consume approximately the same model-state memory?

### My answer

The experiment made this clear to me: more GPUs do not shrink per-GPU model state. Each rank still holds the full parameter, gradient, and Adam tensors, so peak memory stays about the same on every GPU (~16 MB here). Data parallelism splits the **batch**, not the **weights**. `count_replication_factor(..., 'param_element_ids') == 32` matches that — replication factor 32, not 1/32.

## 09. ZeRO-1

In [ ]:
m_z1 = run_strategy('zero1', cfg)
print(m_z1)

**OBSERVATION:**

ZeRO-1 drops `peak_memory_per_gpu` to **8,328,884** bytes versus baseline **16,078,884**. Gradient all-reduce volume is unchanged at **4,000,000** bytes. On rank 0 the optimizer slice in the breakdown shrinks because `optimizer_element_ids` is only one shard.

**QUESTION:** Which memory component stopped being fully replicated? Why?

### My answer

The experiment made this clear from the ownership map: only **optimizer state** (Adam m and v in FP32) is sharded in ZeRO-1. Parameters and gradients are still full on every GPU, so the step still all-reduces the full gradient tensor (4,000,000 bytes here). Peak memory dropped because each rank holds roughly 1/32 of the Adam state instead of the whole thing.

## 10. ZeRO-2

In [ ]:
m_z2 = run_strategy('zero2', cfg)
print(m_z2)

**OBSERVATION:**

ZeRO-2 peak memory is **6,391,384** bytes. Communication switches to reduce-scatter with **2,000,000** bytes logged for the step (versus **4,000,000** all-reduce at baseline). `grad_element_ids` and `optimizer_element_ids` are partitioned per rank; `param_element_ids` stays full.

**QUESTION:** Why does sharding gradients save memory but not remove parameter replication?

### My answer

In this simulator, forward/backward still need a full weight tensor locally, so `param_element_ids` stays full on every rank. After backward, reduce-scatter leaves each rank with only its gradient shard; optimizer state stays sharded like ZeRO-1. Parameter memory did not drop — only gradients and optimizer did (peak **6,391,384** bytes vs **8,328,884** at ZeRO-1).

## 11. ZeRO-3

In [ ]:
m_z3 = run_strategy('zero3', cfg)
print(m_z3)

**OBSERVATION:**

ZeRO-3 aggregate `memory_per_gpu` is only **578,884** bytes steady-state, but `peak_memory_per_gpu` is **2,578,884** bytes when the temporary full parameter view is counted. `communication_bytes` is **6,000,000**, above baseline **4,000,000**. Per `zero3.py`: all-gather parameter shards when the full view is required, reduce-scatter gradients into shards, then all-gather parameter shards again after the local optimizer update.

**QUESTION:** Why does memory decrease further? Where did additional communication come from?

### My answer

Steady-state memory falls because parameters, gradients, and optimizer are all sharded — each rank’s `param_element_ids` covers 1/32 of the elements. The extra **6,000,000** communication bytes vs **4,000,000** baseline is the cost of that layout: all-gather parameter shards when the full parameter view is required; reduce-scatter gradients back into shards. Peak memory spikes above steady state because `temp_full_param_elements` models the gathered view, not the quiet shard footprint.

## 12. Communication simulation

In [ ]:
from src.communication import all_reduce, reduce_scatter, all_gather
ranks = list(range(32))
b = cfg.parameter_count * cfg.bytes_per_param_train
for op in [all_reduce(cfg, ranks, b), reduce_scatter(cfg, ranks, b), all_gather(cfg, ranks, b//32)]:
    print(op.operation, op.bytes_moved, op.label())

## 13. Compute simulation

In [ ]:
from src.compute import step_compute, simulated_time_from_work
cw = step_compute(model, cfg)
print('compute_work', cw.total_work)
print('simulated_compute_time', simulated_time_from_work(cw.total_work, cfg))

## 14. Communication/compute overlap (conceptual)

In [ ]:
from src.communication import overlap_adjusted_time
t_comm = m_z3.communication_time
t_comp = m_z3.compute_time
print('overlap-adjusted comm time', overlap_adjusted_time(t_comm, t_comp, cfg.bucket_size_bytes, b, cfg.overlap_fraction))

## 15. Experiment matrix

In [ ]:
records = run_all_experiments(cfg)
scaling = scaling_sweep(cfg)
out = Path('outputs/experiment_matrix.json')
save_records(records + scaling, out)
print('saved', out, 'rows', len(records)+len(scaling))

## 16. Results

In [ ]:
import pandas as pd
df = pd.DataFrame([r.__dict__ for r in records])
df[['experiment_id','strategy','peak_memory_per_gpu','communication_bytes','total_step_time','communication_fraction']]

## 17. Plots

In [ ]:
fig_dir = Path('outputs/figures')
generate_all_plots(records + scaling, fig_dir)
list(fig_dir.glob('*.png'))

In [ ]:
from IPython.display import Image, display
for name in ['peak_memory_per_gpu.png','memory_breakdown.png','communication_volume.png','compute_vs_comm.png']:
    display(Image(filename=fig_dir / name))

## 18. Sanity checks

In [ ]:
from src.virtual_gpu import (
    assign_replicated, assign_zero1_optimizer_shard, assign_zero2_grad_opt_shard,
    assign_zero3_full_shard, count_replication_factor, sum_shard_coverage, make_cluster,
)
total = cfg.parameter_count
gpus = make_cluster(cfg)
assign_replicated(gpus, total)
assert count_replication_factor(gpus, 'param_element_ids', total) == 32
assign_zero1_optimizer_shard(gpus, total, 32)
assert count_replication_factor(gpus, 'optimizer_element_ids', total) == 1
assign_zero2_grad_opt_shard(gpus, total, 32)
assert count_replication_factor(gpus, 'grad_element_ids', total) == 1
assign_zero3_full_shard(gpus, total, 32)
assert sum_shard_coverage(gpus, 'param_element_ids', total) == total
print('sanity checks passed')

## 19. Concept → Evidence

| Concept | Evidence in this notebook |
|--------|---------------------------|
| Data Parallelism | `m_base` peak memory |
| All-reduce | baseline `communication_bytes` |
| ZeRO-1 | optimizer shard ownership |
| ZeRO-2 | reduce-scatter + grad shards |
| ZeRO-3 | all-gather + param shards |
| Overlap | bucket experiment rows + overlap cell |

# What I Understood

The thread through this notebook is **ownership**: baseline DP replicates everything; each ZeRO stage shards one more tensor class until ZeRO-3 shards params, grads, and optimizer. Memory accounting follows those element sets; communication is whatever collectives each strategy calls, with bytes and simulated time from `communication.py` (simulation assumptions, not CUDA timers). The seven-question block and final reflection tie that to the experiment numbers.

## Seven core questions (concept → this lab)

**1. Why doesn't normal data parallelism reduce model-state memory per GPU?**  
Because each rank still stores full weights, full gradients, and full optimizer state (`assign_replicated`). In my run, peak memory stayed ~16 MB per GPU from world size 1 through 32 on baseline — only aggregate cluster memory grows.

**2. What exactly does ZeRO-1 shard?**  
Only `optimizer_element_ids` — Adam m/v per parameter element. Params and grads stay replicated; all-reduce size stayed 4,000,000 bytes.

**3. What does ZeRO-2 add?**  
Gradient sharding via reduce-scatter: `grad_element_ids` is partitioned. Params are still full replicas; comm dropped to 2,000,000 bytes in the log.

**4. Why does ZeRO-3 require all-gather?**  
Forward/backward need a full parameter view, but steady storage is a shard. `zero3.py` all-gathers parameter shards when that full view is required (before compute and again after the local optimizer update).

**5. Why does ZeRO-3 require reduce-scatter?**  
Gradients are sharded too — after backward we reduce-scatter the full gradient tensor so each rank keeps only its shard for the local optimizer update.

**6. Why can lower memory mean higher communication?**  
The ZeRO-3 step logged 6,000,000 communication bytes vs 4,000,000 for baseline: all-gather parameter shards when the full view is required; reduce-scatter gradients back into shards — extra collectives instead of storing full weights on every rank.

**7. What is the overall memory vs communication trade-off?**  
ZeRO-3 cut steady per-GPU model-state memory to ~0.55 MB (aggregate accounting) but raised bytes moved and introduced a higher **peak** (~2.58 MB) during gathers. You buy headroom with sharding; you pay in collective traffic and transient buffers — still only a simulation, not a GPU benchmark.

## Reflection questions (summary)

**OBSERVATION:**

Multi-node ZeRO-3 (`exp6`) reports `communication_time` **0.000384** s vs single-node assumption (`exp5`) **0.000154** s for the same **6,000,000** bytes. Baseline scaling: `total_step_time` goes from **0.624** s at world size 1 to **0.62464** s at 32 while `peak_memory_per_gpu` stays ~16 MB.

**QUESTION:** Why can adding GPUs fail to provide proportional speedup?

### My answer

The scaling rows show `compute_work` per rank does not shrink as world size grows. `total_step_time` moves from **0.624** s (world size 1) to **0.62464** s (32 GPUs) while `peak_memory_per_gpu` stays ~16 MB on baseline. Communication time ticks up; exp6 is slower than exp5 for the same **6,000,000** bytes when inter-node bandwidth applies. Here comm is tiny vs compute, so the curve is almost flat — not a proportional speedup from adding GPUs alone.

## 22. Final observations

All metrics above come from `outputs/experiment_matrix.json` produced in-section (default `SimConfig`, 1M parameters, 32 ranks). Re-run `python scripts/run_all.py` after changing code or config.

## 23. Reproducibility

```bash
cd session12
python3 -m venv .venv && source .venv/bin/activate
pip install -r requirements.txt
python scripts/run_all.py
```

## Interactive summary tables

In [ ]:
summary = pd.DataFrame([
    {'strategy': m.strategy, 'peak_memory': m.peak_memory_per_gpu, 'comm_bytes': m.communication_bytes,
     'compute_t': m.compute_time, 'total_t': m.total_step_time}
    for m in [m_base, m_z1, m_z2, m_z3]
])
summary

| Question | Evidence | My explanation |
|----------|----------|----------------|
| Why does ZeRO-1 save memory? | Peak 8,328,884 vs 16,078,884 bytes | Optimizer state is sharded; params/grads still full. |
| Why does ZeRO-2 save more? | Peak 6,391,384 bytes | Gradients shard too; reduce-scatter replaces full grad replication. |
| Why does ZeRO-3 save more again? | Steady ~578,884 B/GPU aggregate slice | Params shard at rest; only gather windows need full tensors. |
| Why does communication increase? | 6,000,000 vs 4,000,000 bytes | All-gather parameter shards when full view needed; reduce-scatter grads to shards (`zero3.py`). |
| Why does multi-node matter? | exp5 vs exp6 comm time | Same bytes, higher simulated time when inter-node bandwidth applies. |
| Why does bucket size matter? | `overlap_adjusted_time` + exp7 | Smaller buckets expose more overlap opportunities in the conceptual model; comm bytes unchanged. |
| Why doesn't more GPU always speed up? | scaling_ws* `total_step_time` | Per-rank compute unchanged; comm and replication do not vanish. |

## My Understanding — Final Reflection

Tracing **who owns each element** on the 32 virtual GPUs made ZeRO stages feel concrete instead of abstract. Baseline DP: every rank’s map covers all one million parameter elements — peak ~16 MB/GPU whether world size is 1 or 32 in the matrix; only aggregate memory grows. ZeRO-1 peak ~8.3 MB (optimizer sharded); ZeRO-2 ~6.4 MB (gradients too); ZeRO-3 steady ~0.55 MB/GPU in the aggregate row but **6,000,000** comm bytes vs **4,000,000** baseline.

The experiment made the steady vs peak split on ZeRO-3 obvious: ~**578,884** bytes steady vs ~**2,578,884** bytes peak when `temp_full_param_elements` models the gather window. You all-gather parameter shards when the full parameter view is required; reduce-scatter gradients back into shards — lower quiet memory, more collective traffic.

Bandwidth and step times here are **simulation assumptions** in `communication.py` (useful for exp5 vs exp6 topology, not NVIDIA benchmarks). The useful part is the chain: ownership sets in code → experiment JSON → interpretation.